# Step 19.2 — Train / Validation Loop

本节覆盖 periodic validation loss 与 train/eval behavior，沿用19.1六项候选、完整MiniGPT、sampler、标量loss、Adam和固定MLX v0.32.2接口。

19.1提交 `3d2e95606daf1b5d789599c3d29fd0696240be26` 已独立核心静态验收PASS，实际修正0/2。这是材料验收，不是学员掌握或本机已有语料的证明。

所有代码单元均未执行且无输出。本节初始修正0/2；完成即停在19.2，不进入19.3。

## 1. 自包含依赖与候选配置

工作副本必须恢复所有导入、19.1候选、数据读取/切分/sampler、完整MiniGPT定义、标量loss、验证helper与循环。不能依赖旧kernel或假定本地语料已存在。缓存默认 `data/tinyshakespeare/input.txt`；下载入口必须单独显式启用，默认关闭。项目固定 `mlx==0.32.2`。

候选保持：width=64、layers=2、heads=4、block_size=64、batch_size=4、learning_rate=3e-4，`max_seq_len=block_size`。真实V必须由实际字符表获得。

In [ ]:
import math
import random
from pathlib import Path
from urllib.request import urlopen
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

baseline_candidate = {"model_width":64, "layers":2, "heads":4, "block_size":64, "batch_size":4, "learning_rate":3e-4}
loop_plan = {"num_updates":4, "eval_every":2, "val_batches":2}
seeds = {"model":2028, "train":2026, "validation":2027}

def make_model_kwargs(c, vocab_size):
    return {"vocab_size":vocab_size, "dims":c["model_width"], "num_heads":c["heads"], "num_blocks":c["layers"], "max_seq_len":c["block_size"]}
def make_sampler_kwargs(c): return {"batch_size":c["batch_size"], "block_size":c["block_size"]}
def make_optimizer_kwargs(c): return {"learning_rate":c["learning_rate"], "betas":[0.9,0.999], "eps":1e-8, "bias_correction":True} # explicit course choice; MLX default False

## 2. 数据边界与Sampler

先将连续token stream按90/10切分，再将 `train_tokens` 与 `val_tokens` 分别传给sampler。每个样本取T+1个连续token构造X/Y，各为(B,T)；窗口不跨split边界。Y已shift，loss中不再shift。两侧都必须至少有T+1=65个token。

训练使用持久Python RNG；验证每次使用独立固定seed重建RNG，以复用相同抽样窗口，不消耗训练RNG。它是抽样估计，不代表完整遍历validation split。

In [ ]:
SOURCE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
def acquire_tiny_shakespeare(path, source_url=SOURCE_URL):
    path=Path(path)
    if path.is_file(): return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=30) as response: raw=response.read()
    tmp=path.with_suffix('.tmp'); tmp.write_bytes(raw); tmp.replace(path); return path
def load_cached_character_corpus(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f"Cache missing: {path.resolve()}; correct path or explicitly acquire it.")
    text=path.read_text(encoding='utf-8'); chars=sorted(set(text)); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for i,ch in enumerate(chars)}
    return text, chars, stoi, itos, mx.array([stoi[ch] for ch in text], dtype=mx.int32)
def split_train_validation(stream, fraction=0.90):
    if len(stream.shape)!=1 or stream.shape[0]<2 or not 0<fraction<1: raise ValueError('Expected 1-D stream and 0<fraction<1')
    s=int(stream.shape[0]*fraction)
    if s<=0 or s>=stream.shape[0]: raise ValueError('Empty split')
    return stream[:s],stream[s:],s
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape)!=1 or type(batch_size) is not int or batch_size<1 or type(block_size) is not int or block_size<1: raise ValueError('Invalid sampler inputs')
    n=data.shape[0]
    if n<block_size+1: raise ValueError('Split needs block_size+1 tokens')
    starts=[rng.randrange(0,n-block_size) for _ in range(batch_size)]; xs=[]; ys=[]
    for s in starts:
        chunk=data[s:s+block_size+1]; xs.append(chunk[:-1]); ys.append(chunk[1:])
    return mx.stack(xs,axis=0),mx.stack(ys,axis=0),starts

## 3. 完整模型与loss

模型定义保持既有15.4母版的CausalMultiHeadAttention、FeedForwardMLP、TransformerBlock与MiniGPT。恢复时应将这四个类的完整定义复制到本节工作副本，不使用SharedLogitModel代替；架构保持Pre-Norm RMSNorm、因果attention、C→4C→C MLP、独立Block、final RMSNorm和untied LM head。

Loss使用raw logits (B,T,V)，flatten为(B*T,V)与(B*T,)，逐位置cross-entropy后mean为标量()。训练调用nn.value_and_grad；验证直接调用loss_fn。

In [ ]:
# Insert the complete, unchanged CausalMultiHeadAttention, FeedForwardMLP,
# TransformerBlock and MiniGPT definitions from the accepted Step 15.4 master here.
# This cell is intentionally a definition-restoration checkpoint, not an executable placeholder.

def loss_fn(model, X, Y):
    logits=model(X); B,T,V=logits.shape; n=B*T
    flat_logits=mx.reshape(logits,(n,V)); flat_targets=mx.reshape(Y,(n,))
    losses=nn.losses.cross_entropy(flat_logits,flat_targets,axis=-1,reduction='none')
    return mx.mean(losses)

## 4. Train/eval、梯度与求值

`model.train()`设置training mode；`model.eval()`等价于`model.train(False)`。它不关闭梯度、不冻结参数、不自动完成前向。梯度由是否调用梯度变换决定，更新由是否调用optimizer决定。

当前MiniGPT没有Dropout、BatchNorm或读取`self.training`的前向分支；现有Embedding/Linear/GELU/RMSNorm不按模式改变公式。因此相同参数与输入下，单纯切换模式不改变本模型的前向数学定义。

`mx.eval`要求选定数组及依赖完成并返回None；它不是模式切换，也不等于关闭梯度。验证helper不接收optimizer、不调用nn.value_and_grad，因此没有参数更新路径。

In [ ]:
def estimate_validation_loss(model, val_tokens, batch_size, block_size, num_batches, seed):
    if type(num_batches) is not int or num_batches<1: raise ValueError('num_batches must be positive int')
    rng=random.Random(seed); was_training=model.training; model.eval()
    try:
        total_loss=0.0; total_positions=0
        for _ in range(num_batches):
            Xv,Yv,_=get_batch(val_tokens,batch_size,block_size,rng)
            loss=loss_fn(model,Xv,Yv) # no gradient transform, no optimizer
            mx.eval(loss); n=Yv.shape[0]*Yv.shape[1]
            total_loss += loss.item()*n; total_positions += n
        return total_loss/total_positions
    finally:
        model.train(was_training)

## 5. 周期性验证与参数版本

第k次训练调用返回的是更新前的 `L_train(θ[k−1]; B[k])`；更新后模型持有θ[k]，Adam状态也前进一步。到验证周期时，所有validation batch都使用同一当前参数θ[k]，得到`L_val(θ[k])`。不能把训练步返回的旧参数loss当成更新后loss。

若第j批mean loss为Lⱼ、监督位置数为nⱼ，合并平均为 `sum(nⱼ Lⱼ)/sum(nⱼ)`。本节固定B/T，因此每批nⱼ=256，等权batch mean的平均与位置加权平均等价。纸面例：两个假设mean为1.2和1.8，平均1.5；这不是模型结果。

In [ ]:
def run_training_with_validation(model, optimizer, train_step, training_state,
                                 train_tokens, val_tokens, train_rng, batch_size, block_size,
                                 num_updates, eval_every, val_batches, val_seed):
    if type(num_updates) is not int or num_updates<1 or type(eval_every) is not int or eval_every<1: raise ValueError('Invalid loop plan')
    model.train(); mx.eval(training_state); completed=int(optimizer.state['step'].item())
    initial_val=estimate_validation_loss(model,val_tokens,batch_size,block_size,val_batches,val_seed)
    history=[{'completed_updates':completed,'train_loss_before_update':None,'validation_loss_current':initial_val}]
    for local_update in range(1,num_updates+1):
        model.train(); X,Y,_=get_batch(train_tokens,batch_size,block_size,train_rng)
        train_loss_before_update=train_step(X,Y)
        mx.eval(train_loss_before_update,training_state); completed += 1
        train_value=train_loss_before_update.item(); val_value=None
        if completed%eval_every==0 or local_update==num_updates:
            val_value=estimate_validation_loss(model,val_tokens,batch_size,block_size,val_batches,val_seed)
        history.append({'completed_updates':completed,'train_loss_before_update':train_value,'validation_loss_current':val_value})
    return history

## 6. 编译状态与未来使用顺序

将模型、nn.value_and_grad、Adam、`[model.state, optimizer.state]`原容器引用和compiled callable作为同一匹配setup。训练图只执行forward/loss、梯度与Adam update并返回更新前loss；采样、模式切换、日志和mx.eval留在图外。编译不等于求值，也不保证提速或省内存。

未来work copy操作顺序：恢复本Notebook中的完整MiniGPT类定义；定位真实缓存，必要时主动启用可选获取；执行本地bootstrap；构造model与optimizer并初始化状态；定义训练图和compiled wrapper；最后才执行训练循环。当前没有loss记录、训练结果或验证结果。

累计修正：16.1/16.2各1/2；16.3—16.5、17.1—17.5、18.1—18.5及19.1各0/2；19.2初始0/2。只编写与发布材料，本轮不执行Notebook、训练、验证、测试、采样、测量或语料读写。停在19.2，不进入19.3；后续逐小节至大Step20完成后停止，不进入21。